# SAURIA + Gather full environment notebook

Questo notebook crea una workload SAURIA, genera gli stimuli vanilla tramite `sauria_lib.py`, poi inserisce automaticamente gli stimuli gather in modalità **DRAM writeback**:

```text
gather legge dense/CSR/idx da DRAM esterna
gather scrive IFMAP gathered in DRAM esterna
SAURIA parte normalmente e carica IFMAP con il suo DMA standard
```

Questa versione non modifica `gather_frontend_axi`; richiede però che in `sauria_subsystem.sv` i canali gather siano instradati così durante `gather_busy`:

```text
gather_ext_mem.AR/R       -> DRAM esterna
gather_sauria_mem.AW/W/B  -> DRAM esterna
dma_ext_mem               -> DRAM esterna quando gather_busy=0
```


In [1]:
from pathlib import Path
import shutil
import subprocess
import sys
import os
import numpy as np
import torch
import dotenv

# Adjust these if your notebook is not run from Python/notebooks.
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parents[1] if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SAURIA_ROOT = REPO_ROOT  # Backward-compatible alias used by some debug cells.
PYTHON_DIR = REPO_ROOT / "Python"
TEST_DIR = REPO_ROOT / "test"
VERILATOR_DIR = TEST_DIR / "verilator"
STIMULI_DIR = TEST_DIR / "stimuli"
VANILLA_STIMULI_DIR = TEST_DIR / "stimuli_vanilla_for_gather"
GATHER_STIMULI_DIR = TEST_DIR / "stimuli"

dotenv.load_dotenv(REPO_ROOT / "env", override=True)
sys.path.insert(1, str(PYTHON_DIR))
# Put sauria_gather_env.py in Python/src or in the same folder as this notebook.
sys.path.insert(1, str(NOTEBOOK_DIR))
sys.path.insert(1, str(PYTHON_DIR / "src"))

import src.hw_versions as hwv
import src.sauria_lib as slib
import sauria_gather_env as genv

print("REPO_ROOT:", REPO_ROOT)
print("STIMULI_DIR:", STIMULI_DIR)


REPO_ROOT: /home/henry/sauria
STIMULI_DIR: /home/henry/sauria/test/stimuli


## 1. Hardware version

Usa la stessa versione che hai compilato in Verilator.


In [2]:
sauria_version = "FP16_8x16_AXI64"

cwd = os.getcwd()

os.chdir("../../test/verilator")
f1 = open("verilator_compile.log","w")
subprocess.call(["sh","./compile_sauria.sh",sauria_version],stdout=f1)
os.chdir(cwd)

HW_PARAMS = hwv.get_params(sauria_version)
print(HW_PARAMS)


{'MainMemory_offset': 0, 'SAURIA_offset_DMA': 3489660928, 'CTRL_offset': 1073741824, 'CORE_offset': 1342177280, 'DMA_offset': 1610612736, 'CFG_CON_offset': 512, 'CFG_IFM_offset': 1024, 'CFG_WEI_offset': 1536, 'CFG_PSM_offset': 2048, 'MEMA_offset': 262144, 'MEMB_offset': 524288, 'MEMC_offset': 786432, 'CFG_AXI_DATA_WIDTH': 32, 'CFG_AXI_ADDR_WIDTH': 32, 'MEMA_DEPTH': 2048, 'MEMB_DEPTH': 1024, 'MEMC_DEPTH': 2048, 'DATA_AXI_DATA_WIDTH': 64, 'DATA_AXI_ADDR_WIDTH': 32, 'X': 16, 'Y': 8, 'DILP_W': 64, 'PARAMS_W': 8, 'TH_W': 2, 'IFM_FIFO_POSITIONS': 5, 'WEI_FIFO_POSITIONS': 4, 'FIFO_FILL_CYCLES': 1, 'IA_W': 16, 'IB_W': 16, 'OC_W': 16, 'OP_TYPE': 1, 'IA_MANT': 10, 'IB_MANT': 10, 'IC_MANT': 10, 'rounding': 'RNE', 'approx_comp': False, 'mul_type': 3, 'M': 14, 'add_type': 4, 'A': 16, 'MEMA_W': 128, 'MEMB_W': 256, 'MEMC_W': 128, 'ADRA_W': 11, 'ADRB_W': 10, 'ADRC_W': 11, 'MEMA_N': 8, 'IFM_WOFS_W': 3, 'IFM_IDX_W': 15, 'MEMB_N': 16, 'WEI_WOFS_W': 4, 'WEI_IDX_W': 15, 'MEMC_N': 8, 'PSM_WOFS_W': 3, 'PSM_I

## 2. Workload minima legale per debug gather

Per `FP16_8x16`, `get_conv_dict` richiede:

```text
Cw_tile % Y_used == 0
C_out_tile % X_used == 0
```

Quindi il caso piccolo sotto usa `C_out_tile=16` e `Cw_tile=8`.


In [3]:
# Minimal legal smoke-test convolution.
C_in = 64
C_out = 16
Kh, Kw = 1, 1
s = 1
d = 1

B_conv_torch = torch.nn.Conv2d(C_in, C_out, (Kh, Kw), stride=s, dilation=d, bias=False)

Cw = 8
Ch = 1
Aw = (1 + s * (Cw - 1)) + (1 + d * (Kw - 1)) - 1
Ah = (1 + s * (Ch - 1)) + (1 + d * (Kh - 1)) - 1

tensor_A_torch = torch.randn(C_in, Ah, Aw, dtype=torch.float32)
tensor_C_torch = B_conv_torch(tensor_A_torch)

tensor_A = np.array(tensor_A_torch.detach())
tensor_B = np.array(B_conv_torch.weight.detach())
preload_C = np.zeros([C_out, Ch, Cw])
tensor_C = np.array(tensor_C_torch.detach())

print("A:", tensor_A.shape)
print("B:", tensor_B.shape)
print("C:", tensor_C.shape)
print("IFMAP values:", C_in * Ah * Aw)


A: (48, 1, 8)
B: (16, 48, 1, 1)
C: (16, 1, 8)
IFMAP values: 384


/tmp/ipykernel_684/3176052864.py:18: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  tensor_A = np.array(tensor_A_torch.detach())
/tmp/ipykernel_684/3176052864.py:19: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  tensor_B = np.array(B_conv_torch.weight.detach())
/tmp/ipykernel_684/3176052864.py:21: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn mo

In [4]:
tensor_shapes = [tensor_A.shape, tensor_B.shape, tensor_C.shape]

TILING_DICT = {
    "C_tile_shape": [16, 1, 8],   # [C_out_tile, Ch_tile, Cw_tile]
    "tile_cin": 64,
    "X_used": 16,
    "Y_used": 8,
}

CONV_DICT = slib.get_conv_dict(tensor_shapes, TILING_DICT, HW_PARAMS, d=d, s=s, preloads=False)
print(CONV_DICT)


{'B_w': 1, 'B_h': 1, 'C_w': 8, 'C_h': 1, 'C_c': 16, 'A_w': 8, 'A_h': 1, 'A_c': 48, 'AB_c': 48, 'd': 1, 's': 1, 'w_til': 8, 'h_til': 1, 'c_til': 48, 'k_til': 16, 'A_w_til': 8, 'A_h_til': 1, 'X_ext_tiles': 1, 'Y_ext_tiles': 1, 'K_ext_tiles': 1, 'C_ext_tiles': 1, 'N_total_tiles': 1, 'B_w_eff': 1, 'B_h_eff': 1, 'X_int_tiles': 1, 'Y_int_tiles': 1, 'K_int_tiles': 1, 'N_cswitch': 1, 'X_used': 16, 'Y_used': 8, 'preload_en': False, 'Dil_pat': 9223372036854775808, 'rows_active': 255, 'cols_active': 65535, 'lwoffs': array([0, 1, 2, 3, 4, 5, 6, 7]), 'thres': 0}


## 3. Genera stimuli vanilla SAURIA

Questa cella usa `sauria_lib.py` come nel notebook originale. Prima verifica che il test vanilla passi.

`generate_vcd=False` evita VCD enormi.


In [5]:
SAURIA_outputs, SAURIA_stats = slib.Conv2d_SAURIA(
    tensor_A,
    tensor_B,
    preload_C,
    tensor_C,
    CONV_DICT,
    HW_PARAMS,
    generate_vcd=False,
    print_statistics=True,
    silent=False,
)

# Salva una copia degli stimuli vanilla appena generati.
VANILLA_STIMULI_DIR.mkdir(parents=True, exist_ok=True)
genv.copy_stimuli_dir(STIMULI_DIR, VANILLA_STIMULI_DIR)
print("Copied vanilla stimuli to", VANILLA_STIMULI_DIR)



              TEST PASSED

****************************************
          SAURIA STATISTICS
****************************************
Total cycles:				848
Total operations:			12288
Average Throughput:			14.49 OP/cycle (5.66 %)

Number of tiles:			1
Core stall cycles:			89 (10.50 %)
Memory/CGF stall cycles:		706 (83.25 %)

SAURIA memory capacity (A|B|C):		32.0 | 32.0 | 32.0 [kB]
Utilized memory:			0.75 | 1.5 | 0.25 [kB] (2.34 | 4.69 | 0.78 [%])
Copied vanilla stimuli to /home/henry/sauria/test/stimuli_vanilla_for_gather


## 4. Scegli gli indirizzi DRAM IFMAP per il gather

Per il mini-test generato sopra, SAURIA ha `Number of tiles: 1`, quindi usa un solo indirizzo DRAM IFMAP.

Non usare automaticamente un vecchio `sauria_debug.log`: se il log è della workload grande, il notebook genererebbe centinaia di gather e potrebbe anche sovrascrivere la metadata.


In [6]:
# Valori del tile IFMAP della workload minimal generata dal notebook.
N_VALUES = C_in * Ah * Aw          # 16 * 1 * 8 = 128
N_DENSE_COLS = N_VALUES
TILE_BYTES = N_VALUES * 2          # FP16 = 2 byte

# Mini-test: un solo tile IFMAP, letto da DRAM base 0x0 nella configurazione standard.
tile_out_bases = [0x0]

print("N_VALUES:", N_VALUES)
print("TILE_BYTES:", TILE_BYTES)
print("tile_out_bases:", [hex(x) for x in tile_out_bases])

# Solo se vuoi usare un trace nuovo e sei sicuro che sia della workload minimal, abilita questo blocco.
USE_TRACE_FOR_TILE_ADDRS = False
DEBUG_LOG = VERILATOR_DIR / "sauria_debug_minimal.log"

if USE_TRACE_FOR_TILE_ADDRS:
    tiles = genv.extract_ifmap_logical_tiles(
        DEBUG_LOG,
        ifmap_region_start=0x0,
        ifmap_region_end=0x10000,
        tile_bytes=TILE_BYTES,
    )
    genv.write_ifmap_tiles_csv(TEST_DIR / "ifmap_tiles_from_log.csv", tiles)
    tile_out_bases = sorted({t["dram_start_addr"] for t in tiles})
    print("Logical IFMAP reads:", len(tiles))
    print("Unique DRAM tile addresses:", len(tile_out_bases))
    print("First addresses:", [hex(x) for x in tile_out_bases[:8]])


N_VALUES: 384
TILE_BYTES: 768
tile_out_bases: ['0x0']


In [7]:
# Uses genv imported in the setup cell.
RESTORE_CHECK_DIR = REPO_ROOT / "test" / "stimuli_gather_restore_check"

manifest = genv.make_gather_restore_check_stimuli(
    in_dir=VANILLA_STIMULI_DIR,
    out_dir=RESTORE_CHECK_DIR,
    tile_out_base=0x0,
    n_values=N_VALUES,
    n_dense_cols=N_DENSE_COLS,
    dense_stage_base=0x90000,
    poison_out=True,
)

print(manifest)


{'mode': 'gather_restore_check_only', 'tile_out_base': '0x00000000', 'n_values': 384, 'n_dense_cols': 384, 'check_start': '0x00000000', 'check_end_excl': '0x00000300', 'dense_stage_base': '0x00090000', 'comp_base': '0x00091000', 'idx_base': '0x00092000', 'metadata': {'comp_total_beats': 2, 'idx_total_beats': 193, 'idx_natural_beats': 192}, 'poison_out': True}


## 5. Genera stimuli SAURIA + gather

Questa cella prende gli stimuli vanilla e crea una nuova `test/stimuli` con il gather preposto a SAURIA.

Per la workload minima sopra, l’IFMAP teorico è `128` FP16. Però se dal trace vedi che SAURIA legge tile da `680` byte, usa `n_values=340` e gli indirizzi estratti dal log.


In [8]:
# Generate gather+SAURIA stimuli.
# Do not pass fixed comp_base/idx_base: the v2 helper places metadata after the dense staging area
# and checks for overlap automatically.
manifest = genv.make_gather_dram_writeback_stimuli(
    in_dir=VANILLA_STIMULI_DIR,
    out_dir=GATHER_STIMULI_DIR,
    tile_out_bases=tile_out_bases,
    n_values=N_VALUES,
    n_dense_cols=N_DENSE_COLS,
    dense_stage_base=0x90000,
    dense_stage_stride=0x1000,
    idx_extra_beats=1,
    poison_out=True,
    clear_keep_a_bit=True,
    insert_position="prepend",
    deduplicate_tiles=True,
)

print("Generated gather stimuli:")
print(manifest)


Generated gather stimuli:
{'mode': 'gather_dram_writeback_then_sauria_dma', 'num_unique_gather_tiles': 1, 'n_values': 384, 'n_dense_cols': 384, 'dense_stage_base': 589824, 'dense_stage_stride': 4096, 'comp_base': 593920, 'idx_base': 598016, 'idx_extra_beats': 1, 'metadata': {'comp_total_beats': 2, 'idx_total_beats': 193, 'idx_natural_beats': 192}, 'poison_out': True, 'poison_value': 0, 'clear_keep_a_bit': True, 'insert_position': 'prepend', 'input_dir': '/home/henry/sauria/test/stimuli_vanilla_for_gather', 'output_dir': '/home/henry/sauria/test/stimuli', 'tile_out_bases': ['0x00000000']}


## 6. Run Verilator su gather + SAURIA

Esegui senza VCD per il primo controllo. Il risultato atteso è:

```text
Benchmark passed with no errors.
SUCCESS!
```


In [9]:
cmd = ["./Test-Sim", "+debug", "+max-cycles=20000000"]
print("Running:", " ".join(cmd))
res = subprocess.run(cmd, cwd=VERILATOR_DIR, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(res.stdout[-5000:])
print("returncode:", res.returncode)


Running: ./Test-Sim +debug +max-cycles=20000000
91000 into address 7000001C
[220] CFG idx 4
[230] CFG idx 5
Writing 2 into address 70000020
[240] CFG idx 5
[250] CFG idx 6
Writing 92000 into address 70000024
[260] CFG idx 6
[270] CFG idx 7
Writing C1 into address 70000028
[280] CFG idx 7
[290] CFG idx 8
Writing 0 into address 7000002C
[300] CFG idx 8
[310] CFG idx 9
Writing 17F into address 70000030
[320] CFG idx 9
[330] CFG idx 10
Writing 0 into address 70000004
[340] CFG idx 10
[350] CFG idx 11
Writing 1 into address 70000000
[360] CFG idx 11
[370] CFG idx 12
[370] [CFG] Waiting for Gather...
[16580] Gather done.
[16590] CFG idx 13
[16600] CFG idx 13
[16610] CFG idx 13
[16620] CFG idx 13
[16630] CFG idx 13
Read 1 from address 70000008
[16640] CFG idx 14
Writing 3 into address 40000008
[16650] CFG idx 14
[16660] CFG idx 15
Writing 0 into address 40000010
[16670] CFG idx 15
[16680] CFG idx 16
Writing 0 into address 40000014
[16690] CFG idx 16
[16700] CFG idx 17
Writing 8 into address 4

## 7. Se vuoi integrare in `sauria_lib.py`

La soluzione pulita è aggiungere a `sauria_lib.py` un wrapper opzionale che, dopo aver generato gli stimuli vanilla e prima di lanciare Verilator, chiama:

```python
genv.make_gather_dram_writeback_stimuli(...)
```

Per ora questo notebook evita modifiche invasive a `sauria_lib.py` e mantiene separati:

```text
SAURIA vanilla generation
Gather patch generation
Verilator run
```


## 8. Benchmark gather dentro SAURIA

Questa sezione legge i registri performance del gather dopo `REG_DONE`, lancia Verilator e costruisce una tabella con cicli, beat, byte e bandwidth effettiva.

Prerequisito RTL: `gather_frontend_axi.sv` deve esporre in AXI-Lite questi registri:

```text
0x70000034 PERF_TOTAL_CYCLES
0x70000038 PERF_RD_CYCLES
0x7000003C PERF_WR_CYCLES
0x70000040 PERF_R_STALL
0x70000044 PERF_W_STALL
0x70000048 PERF_R_BEATS
0x7000004C PERF_W_BEATS
```


In [10]:
import re
import pandas as pd
from IPython.display import display

GATHER_BASE = 0x7000_0000
PERF_REGS = {
    GATHER_BASE + 0x34: "gather_cycles",
    GATHER_BASE + 0x38: "rd_cycles",
    GATHER_BASE + 0x3C: "wr_cycles",
    GATHER_BASE + 0x40: "r_stall",
    GATHER_BASE + 0x44: "w_stall",
    GATHER_BASE + 0x48: "r_beats",
    GATHER_BASE + 0x4C: "w_beats",
}
AXI_BYTES = 8
benchmark_rows = []

def _read_golden_rows(stimuli_dir):
    path = Path(stimuli_dir) / "GoldenStimuli.txt"
    rows = []
    for line in path.read_text().splitlines():
        if line.strip():
            rows.append([int(tok, 16) for tok in line.split()])
    return rows

def _write_golden_rows(stimuli_dir, rows):
    path = Path(stimuli_dir) / "GoldenStimuli.txt"
    with path.open("w") as f:
        for row in rows:
            f.write(" ".join(f"{int(x) & 0xFFFFFFFF:X}" for x in row) + "\n")

def add_gather_perf_reads(stimuli_dir, verbose=True):
    """Insert AXI-Lite reads of gather performance registers before any final gold check.

    In gather-only restore-check stimuli, the REG_DONE read can also be the row
    that triggers the final gold_dram comparison. If perf reads are appended after
    that row, Test-Sim finishes before reaching them.
    """
    stimuli_dir = Path(stimuli_dir)
    rows = _read_golden_rows(stimuli_dir)
    done_addr = GATHER_BASE + 0x08
    perf_addrs = list(PERF_REGS.keys())
    perf_rows = [[0, addr, 0, 1, 0, 0, 0] for addr in perf_addrs]

    def is_perf_row(row):
        return row[1] in PERF_REGS and row[2] == 0 and row[3] == 1

    # Idempotent: remove perf reads if this cell was run before.
    rows = [r for r in rows if not is_perf_row(r)]

    insert_idx = None
    for i, row in enumerate(rows):
        data, addr, we, re = row[0], row[1], row[2], row[3]
        if addr == done_addr and we == 0 and re == 1:
            insert_idx = i + 1
            break

    if insert_idx is None:
        raise RuntimeError("Could not find gather REG_DONE read in GoldenStimuli.txt")

    original_done_row = rows[insert_idx - 1].copy()
    done_row_had_final_check = bool(original_done_row[6])

    # If REG_DONE also triggers final check, clear it and duplicate it after perf reads.
    if done_row_had_final_check:
        rows[insert_idx - 1][6] = 0
        final_check_row = original_done_row
    else:
        final_check_row = None

    rows = (
        rows[:insert_idx]
        + perf_rows
        + ([final_check_row] if final_check_row is not None else [])
        + rows[insert_idx:]
    )

    _write_golden_rows(stimuli_dir, rows)

    if verbose:
        print(f"Inserted {len(perf_rows)} performance reads in {stimuli_dir}/GoldenStimuli.txt")
        if done_row_had_final_check:
            print("Moved gather-only final gold check after performance reads.")

def install_stimuli(src_dir, dst_dir=STIMULI_DIR):
    """Copy a generated stimulus directory into test/stimuli for Test-Sim.

    If src_dir is already test/stimuli, do nothing. This avoids SameFileError
    when the benchmark stimuli are generated directly into STIMULI_DIR.
    """
    src_dir = Path(src_dir).resolve()
    dst_dir = Path(dst_dir).resolve()

    if src_dir == dst_dir:
        print("Stimuli already installed:", dst_dir)
        return

    dst_dir.mkdir(parents=True, exist_ok=True)
    for name in ["GoldenStimuli.txt", "initial_dram.txt", "gold_dram.txt", "tstcfg.txt"]:
        src = src_dir / name
        dst = dst_dir / name
        if src.resolve() == dst.resolve():
            continue
        shutil.copy2(src, dst)

    print("Installed stimuli:", src_dir, "->", dst_dir)

def run_testsim(label, stimuli_dir=None, max_cycles=20_000_000, debug=True, extra_args=None):
    if stimuli_dir is not None:
        install_stimuli(stimuli_dir)

    cmd = ["./Test-Sim"]
    if debug:
        cmd.append("+debug")
    cmd.append(f"+max-cycles={max_cycles}")
    if extra_args:
        cmd.extend(extra_args)

    print("Running:", " ".join(cmd))
    res = subprocess.run(cmd, cwd=VERILATOR_DIR, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    log_text = res.stdout

    log_dir = TEST_DIR / "benchmark_logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / f"{label}.log"
    log_path.write_text(log_text)

    print(log_text[-5000:])
    print("returncode:", res.returncode)
    print("log:", log_path)
    return log_text, res.returncode, log_path

def parse_testsim_log(log_text):
    perf = {name: None for name in PERF_REGS.values()}

    for value_hex, addr_hex in re.findall(r"Read\s+([0-9A-Fa-f]+)\s+from address\s+([0-9A-Fa-f]+)", log_text):
        addr = int(addr_hex, 16)
        if addr in PERF_REGS:
            perf[PERF_REGS[addr]] = int(value_hex, 16)

    gather_done_cycle = None
    m = re.search(r"\[(\d+)\]\s+Gather done", log_text)
    if m:
        gather_done_cycle = int(m.group(1))

    bench_cycle = None
    bench_status = None
    m = re.search(r"\[(\d+)\]\s+Benchmark\s+(passed|failed)", log_text)
    if m:
        bench_cycle = int(m.group(1))
        bench_status = m.group(2)

    timeout = "TIMEOUT" in log_text
    success = "SUCCESS!" in log_text and bench_status == "passed"
    failed = "FAILED!" in log_text or bench_status == "failed"

    r_beats = perf.get("r_beats") or 0
    w_beats = perf.get("w_beats") or 0
    gather_cycles = perf.get("gather_cycles") or 0

    parsed = {
        **perf,
        "gather_done_cycle_from_log": gather_done_cycle,
        "benchmark_cycle": bench_cycle,
        "status": "pass" if success else ("fail" if failed else "unknown"),
        "timeout": timeout,
        "read_bytes": r_beats * AXI_BYTES,
        "write_bytes": w_beats * AXI_BYTES,
        "total_gather_bytes": (r_beats + w_beats) * AXI_BYTES,
        "read_B_per_cycle": (r_beats * AXI_BYTES / gather_cycles) if gather_cycles else None,
        "write_B_per_cycle": (w_beats * AXI_BYTES / gather_cycles) if gather_cycles else None,
        "total_B_per_cycle": ((r_beats + w_beats) * AXI_BYTES / gather_cycles) if gather_cycles else None,
    }
    return parsed

def add_benchmark_row(label, log_text, **meta):
    row = {"label": label, **meta, **parse_testsim_log(log_text)}
    benchmark_rows.append(row)
    df = pd.DataFrame(benchmark_rows)
    display(df)
    out_csv = TEST_DIR / "gather_benchmark_results.csv"
    df.to_csv(out_csv, index=False)
    print("Wrote", out_csv)
    return row


### 8.1 Benchmark gather-only restore-check

Questo misura solo il gather, senza SAURIA compute. Il test è utile per capire bandwidth e stall del gather isolato.


In [11]:
# Generate gather-only restore-check stimuli and insert perf reads.
RESTORE_CHECK_DIR = REPO_ROOT / "test" / "stimuli_gather_restore_check"

manifest_restore = genv.make_gather_restore_check_stimuli(
    in_dir=VANILLA_STIMULI_DIR,
    out_dir=RESTORE_CHECK_DIR,
    tile_out_base=0x0,
    n_values=N_VALUES,
    n_dense_cols=N_DENSE_COLS,
    dense_stage_base=0x90000,
    poison_out=True,
)
print(manifest_restore)

add_gather_perf_reads(RESTORE_CHECK_DIR)
log_text, rc, log_path = run_testsim(
    label=f"gather_only_n{N_VALUES}",
    stimuli_dir=RESTORE_CHECK_DIR,
    max_cycles=20_000_000,
    debug=True,
)
add_benchmark_row(
    f"gather_only_n{N_VALUES}",
    log_text,
    mode="gather_only",
    n_values=N_VALUES,
    n_dense_cols=N_DENSE_COLS,
)


{'mode': 'gather_restore_check_only', 'tile_out_base': '0x00000000', 'n_values': 384, 'n_dense_cols': 384, 'check_start': '0x00000000', 'check_end_excl': '0x00000300', 'dense_stage_base': '0x00090000', 'comp_base': '0x00091000', 'idx_base': '0x00092000', 'metadata': {'comp_total_beats': 2, 'idx_total_beats': 193, 'idx_natural_beats': 192}, 'poison_out': True}
Inserted 7 performance reads in /home/henry/sauria/test/stimuli_gather_restore_check/GoldenStimuli.txt
Moved gather-only final gold check after performance reads.
Installed stimuli: /home/henry/sauria/test/stimuli_gather_restore_check -> /home/henry/sauria/test/stimuli
Running: ./Test-Sim +debug +max-cycles=20000000

Initializing SAURIA test...

Reading stimuli from file...
Test config file ../stimuli/tstcfg.txt has 3 lines.
Stimuli file ../stimuli/GoldenStimuli.txt has 22 lines.

Starting test

[130] CFG idx 0
Writing 3 into address 7000000C
[140] CFG idx 0
[150] CFG idx 1
Writing 3 into address 70000010
[160] CFG idx 1
[170] CFG

,label,mode,n_values,n_dense_cols,gather_cycles,rd_cycles,wr_cycles,r_stall,w_stall,r_beats,...,gather_done_cycle_from_log,benchmark_cycle,status,timeout,read_bytes,write_bytes,total_gather_bytes,read_B_per_cycle,write_B_per_cycle,total_B_per_cycle
0,gather_only_n384,gather_only,384,384,1618,1126,114,829,0,291,...,16580,16991,pass,False,2328,768,3096,1.438813,0.47466,1.913473


Wrote /home/henry/sauria/test/gather_benchmark_results.csv


{'label': 'gather_only_n384',
 'mode': 'gather_only',
 'n_values': 384,
 'n_dense_cols': 384,
 'gather_cycles': 1618,
 'rd_cycles': 1126,
 'wr_cycles': 114,
 'r_stall': 829,
 'w_stall': 0,
 'r_beats': 291,
 'w_beats': 96,
 'gather_done_cycle_from_log': 16580,
 'benchmark_cycle': 16991,
 'status': 'pass',
 'timeout': False,
 'read_bytes': 2328,
 'write_bytes': 768,
 'total_gather_bytes': 3096,
 'read_B_per_cycle': 1.4388133498145859,
 'write_B_per_cycle': 0.4746600741656366,
 'total_B_per_cycle': 1.9134734239802225}

### 8.2 Benchmark gather + SAURIA

Questo misura il flusso completo seriale:

```text
gather -> writeback DRAM -> SAURIA DMA standard -> SAURIA compute
```


In [12]:
# Generate gather+SAURIA stimuli and insert perf reads after gather done.
manifest_full = genv.make_gather_dram_writeback_stimuli(
    in_dir=VANILLA_STIMULI_DIR,
    out_dir=GATHER_STIMULI_DIR,
    tile_out_bases=tile_out_bases,
    n_values=N_VALUES,
    n_dense_cols=N_DENSE_COLS,
    dense_stage_base=0x90000,
    dense_stage_stride=0x1000,
    idx_extra_beats=1,
    poison_out=True,
    clear_keep_a_bit=True,
    insert_position="prepend",
    deduplicate_tiles=True,
)
print(manifest_full)

add_gather_perf_reads(GATHER_STIMULI_DIR)
log_text, rc, log_path = run_testsim(
    label=f"gather_plus_sauria_n{N_VALUES}",
    stimuli_dir=GATHER_STIMULI_DIR,
    max_cycles=20_000_000,
    debug=True,
)
add_benchmark_row(
    f"gather_plus_sauria_n{N_VALUES}",
    log_text,
    mode="gather_plus_sauria",
    n_values=N_VALUES,
    n_dense_cols=N_DENSE_COLS,
)


{'mode': 'gather_dram_writeback_then_sauria_dma', 'num_unique_gather_tiles': 1, 'n_values': 384, 'n_dense_cols': 384, 'dense_stage_base': 589824, 'dense_stage_stride': 4096, 'comp_base': 593920, 'idx_base': 598016, 'idx_extra_beats': 1, 'metadata': {'comp_total_beats': 2, 'idx_total_beats': 193, 'idx_natural_beats': 192}, 'poison_out': True, 'poison_value': 0, 'clear_keep_a_bit': True, 'insert_position': 'prepend', 'input_dir': '/home/henry/sauria/test/stimuli_vanilla_for_gather', 'output_dir': '/home/henry/sauria/test/stimuli', 'tile_out_bases': ['0x00000000']}
Inserted 7 performance reads in /home/henry/sauria/test/stimuli/GoldenStimuli.txt
Stimuli already installed: /home/henry/sauria/test/stimuli
Running: ./Test-Sim +debug +max-cycles=20000000
 15
[16720] CFG idx 15
[16730] CFG idx 15
Read 466 from address 70000038
[16740] CFG idx 16
[16750] CFG idx 16
[16760] CFG idx 16
[16770] CFG idx 16
[16780] CFG idx 16
Read 72 from address 7000003C
[16790] CFG idx 17
[16800] CFG idx 17
[16810

,label,mode,n_values,n_dense_cols,gather_cycles,rd_cycles,wr_cycles,r_stall,w_stall,r_beats,...,gather_done_cycle_from_log,benchmark_cycle,status,timeout,read_bytes,write_bytes,total_gather_bytes,read_B_per_cycle,write_B_per_cycle,total_B_per_cycle
0,gather_only_n384,gather_only,384,384,1618,1126,114,829,0,291,...,16580,16991,pass,False,2328,768,3096,1.438813,0.47466,1.913473
1,gather_plus_sauria_n384,gather_plus_sauria,384,384,1618,1126,114,829,0,291,...,16580,33951,pass,False,2328,768,3096,1.438813,0.47466,1.913473


Wrote /home/henry/sauria/test/gather_benchmark_results.csv


{'label': 'gather_plus_sauria_n384',
 'mode': 'gather_plus_sauria',
 'n_values': 384,
 'n_dense_cols': 384,
 'gather_cycles': 1618,
 'rd_cycles': 1126,
 'wr_cycles': 114,
 'r_stall': 829,
 'w_stall': 0,
 'r_beats': 291,
 'w_beats': 96,
 'gather_done_cycle_from_log': 16580,
 'benchmark_cycle': 33951,
 'status': 'pass',
 'timeout': False,
 'read_bytes': 2328,
 'write_bytes': 768,
 'total_gather_bytes': 3096,
 'read_B_per_cycle': 1.4388133498145859,
 'write_B_per_cycle': 0.4746600741656366,
 'total_B_per_cycle': 1.9134734239802225}

### 8.3 Report benchmark

La tabella sotto permette di capire se il gather è collo di bottiglia:

- `gather_cycles`: cicli interni start→done del gather.
- `read_bytes/write_bytes`: traffico AXI effettivamente trasferito.
- `*_B_per_cycle`: bandwidth media effettiva.
- `r_stall/w_stall`: backpressure sui canali read/write.


In [13]:
df = pd.DataFrame(benchmark_rows)
if not df.empty:
    preferred_cols = [
        "label", "mode", "status", "timeout", "n_values", "n_dense_cols",
        "gather_cycles", "benchmark_cycle", "read_bytes", "write_bytes", "total_gather_bytes",
        "read_B_per_cycle", "write_B_per_cycle", "total_B_per_cycle",
        "rd_cycles", "wr_cycles", "r_stall", "w_stall", "r_beats", "w_beats",
    ]
    display(df[[c for c in preferred_cols if c in df.columns]])
else:
    print("Nessun benchmark ancora eseguito.")


,label,mode,status,timeout,n_values,n_dense_cols,gather_cycles,benchmark_cycle,read_bytes,write_bytes,total_gather_bytes,read_B_per_cycle,write_B_per_cycle,total_B_per_cycle,rd_cycles,wr_cycles,r_stall,w_stall,r_beats,w_beats
0,gather_only_n384,gather_only,pass,False,384,384,1618,16991,2328,768,3096,1.438813,0.47466,1.913473,1126,114,829,0,291,96
1,gather_plus_sauria_n384,gather_plus_sauria,pass,False,384,384,1618,33951,2328,768,3096,1.438813,0.47466,1.913473,1126,114,829,0,291,96


### 8.4 Sweep opzionale

Per presentare un benchmark convincente, ripeti `gather_only` e `gather+SAURIA` variando `N_VALUES`, densità e dimensione buffer RTL. Questa cella è un template: per ogni valore devi rigenerare la workload/stimuli coerenti.


In [14]:
# Template manuale: modifica N_VALUES/N_DENSE_COLS e riesegui le celle 8.1 e 8.2.
# Esempio di punti interessanti:
# sweep_points = [128, 256, 340, 512, 1024]
#
# Nota: N_VALUES deve essere compatibile con il tile che SAURIA leggerà.
# Per il mini-test corrente: N_VALUES = 128.
# Per tile grandi osservati in trace precedenti: N_VALUES = 340.
